# Smriti — Face Embedding Bridge

Run this notebook on **Kaggle** or **Colab** with a GPU runtime. It starts a FastAPI server that receives batches of 112×112 face crops from your desktop Smriti and returns 512-d embeddings.

**Model must match the desktop.** Smriti's default embedder is **AdaFace** (`adaface_ir101_webface12m.onnx`). If you changed it to `glintr100.onnx` in Settings, flip the URL in cell 2 to the matching model — the bridge's `/health` response advertises which model is loaded and Smriti refuses to send work when they don't match (mismatched models produce embeddings in different metric spaces, which would corrupt clustering).


In [ ]:
!pip install -q fastapi uvicorn nest-asyncio python-multipart onnxruntime-gpu==1.18.0 pyngrok

In [ ]:
# Pick ONE model. Default = AdaFace (matches Smriti's default).
# To use glintr100 instead, set MODEL_NAME='glintr100' and swap the wget URL below.
MODEL_NAME = 'adaface_ir101_webface12m'
MODEL_URL = f'https://huggingface.co/MonsterMMORPG/tools/resolve/main/{MODEL_NAME}.onnx'

!wget -q $MODEL_URL -O /content/model.onnx
!ls -lh /content/model.onnx

In [ ]:
# Verify GPU is available
import onnxruntime as ort
sess = ort.InferenceSession('/content/model.onnx', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
print('GPU:', sess.get_providers())

In [ ]:
from fastapi import FastAPI, UploadFile, File
import numpy as np
from PIL import Image
import io

app = FastAPI()
session = sess  # reuse the verified session

def preprocess(img_bytes: bytes) -> np.ndarray:
    """Decode JPEG, resize to 112x112, normalize to [-1, 1], return [1, 3, 112, 112]."""
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    img = img.resize((112, 112), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32)
    arr = (arr - 127.5) / 127.5  # normalize to [-1, 1]
    arr = np.transpose(arr, (2, 0, 1))  # HWC -> CHW
    return np.expand_dims(arr, axis=0)  # [1, 3, 112, 112]

def l2_normalize(vec: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

@app.post('/embed')
async def embed(files: list[UploadFile] = File(...)):
    """Receive N face crops, return N 512-d embeddings."""
    if not files:
        return {'embeddings': []}
    face_bytes = []
    for f in files:
        face_bytes.append(await f.read())
    batch = np.concatenate([preprocess(b) for b in face_bytes], axis=0).astype(np.float32)
    onnx_inputs = {session.get_inputs()[0].name: batch}
    output = session.run(None, onnx_inputs)[0]
    embeddings = [l2_normalize(output[i]).tolist() for i in range(output.shape[0])]
    return {'embeddings': embeddings}

@app.get('/health')
async def health():
    providers = session.get_providers()
    return {
        'status': 'ok',
        'provider': providers[0] if providers else 'unknown',
        'model': MODEL_NAME,
    }

print(f'FastAPI app ready — serving model {MODEL_NAME!r}')

In [ ]:
# === Tunnel: ngrok (default) ===
# Sign up for free at https://ngrok.com — copy your auth token from the dashboard.
from pyngrok import ngrok
ngrok.set_auth_token('YOUR_TOKEN_HERE')  # <-- paste your ngrok authtoken
public_url = ngrok.connect(8000, 'http')
print(f'BRIDGE URL -> {public_url}')
print('Paste this URL into Smriti Settings -> Cloud face acceleration')

In [ ]:
# === Tunnel: cloudflared (alternative, no account needed) ===
# Uncomment the block below to use cloudflared instead of ngrok.
# The URL prints to cloudflared's stdout — watch the cell output.

# !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
# !chmod +x cloudflared
# import subprocess, threading
# def run_tunnel():
#     subprocess.run(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'], check=False)
# threading.Thread(target=run_tunnel, daemon=True).start()
# print('cloudflared tunnel starting — watch for: https://xxx-xxx-xxx.trycloudflare.com')
# import time; time.sleep(5)

In [ ]:
# === Run the server ===
import nest_asyncio, uvicorn
nest_asyncio.apply()
uvicorn.run(app, host='0.0.0.0', port=8000)